🔹 Topic: Cross-Attention
🧠 Core Idea ()

👉 “One sequence looks at another sequence to get information.”

Self-Attention → looks within itself
Cross-Attention → looks at something else
🔥 Simple Example

👉 Translation task:

Input (Encoder): “I love AI”
Output (Decoder): generating “Je…”

While generating “Je”, the model:
👉 looks back at input sentence using cross-attention

In [1]:
"""
Cross-Attention in Transformers — Complete Implementation
=========================================================
Includes:
  1. Scaled Dot-Product Attention (NumPy, for understanding)
  2. MultiHeadCrossAttention (PyTorch module, production-ready)
  3. DecoderLayer using cross-attention
  4. Flash Attention variant (PyTorch 2.0+)
  5. Quick test / demo
"""

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ──────────────────────────────────────────────────────────────
# 1.  NUMPY: Single-head cross-attention (educational)
# ──────────────────────────────────────────────────────────────

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Computes scaled dot-product attention.

    Args:
        Q : np.ndarray  [T_q,  d_k]
        K : np.ndarray  [T_kv, d_k]
        V : np.ndarray  [T_kv, d_v]
        mask : np.ndarray | None  [T_q, T_kv]  — 0 = ignore, 1 = attend

    Returns:
        output  : [T_q, d_v]
        weights : [T_q, T_kv]
    """
    d_k = Q.shape[-1]
    scores = Q @ K.T / math.sqrt(d_k)          # [T_q, T_kv]

    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)

    # Numerically stable softmax
    scores -= scores.max(axis=-1, keepdims=True)
    weights = np.exp(scores)
    weights /= weights.sum(axis=-1, keepdims=True)

    output = weights @ V                        # [T_q, d_v]
    return output, weights


# ──────────────────────────────────────────────────────────────
# 2.  PYTORCH: Multi-Head Cross-Attention
# ──────────────────────────────────────────────────────────────

class MultiHeadCrossAttention(nn.Module):
    """
    Multi-Head Cross-Attention layer.

    Q  comes from the DECODER (target sequence).
    K, V come from the ENCODER (source / memory sequence).

    Args:
        d_model   : int   — model dimension
        num_heads : int   — number of attention heads
        dropout   : float — dropout probability on attention weights
        bias      : bool  — use bias in linear projections
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        dropout: float = 0.1,
        bias: bool = False,
    ):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.scale = math.sqrt(self.d_k)

        # Separate linear projections
        self.W_q = nn.Linear(d_model, d_model, bias=bias)   # from decoder
        self.W_k = nn.Linear(d_model, d_model, bias=bias)   # from encoder
        self.W_v = nn.Linear(d_model, d_model, bias=bias)   # from encoder
        self.W_o = nn.Linear(d_model, d_model, bias=bias)   # output projection

        self.attn_dropout = nn.Dropout(dropout)

        self._init_weights()

    def _init_weights(self):
        for proj in (self.W_q, self.W_k, self.W_v, self.W_o):
            nn.init.xavier_uniform_(proj.weight)
        nn.init.zeros_(self.W_o.weight)   # zero-init output for stability

    # ----------------------------------------------------------
    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """[B, T, d_model]  →  [B, heads, T, d_k]"""
        B, T, _ = x.shape
        return x.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        """[B, heads, T, d_k]  →  [B, T, d_model]"""
        return x.transpose(1, 2).contiguous().view(x.size(0), -1, self.d_model)

    # ----------------------------------------------------------
    def forward(
        self,
        query: torch.Tensor,                            # [B, T_q,  d_model]
        key:   torch.Tensor,                            # [B, T_kv, d_model]
        value: torch.Tensor,                            # [B, T_kv, d_model]
        key_padding_mask: torch.BoolTensor | None = None,  # [B, T_kv] True=ignore
        return_weights: bool = False,
    ):
        """
        Returns:
            output       : [B, T_q, d_model]
            attn_weights : [B, heads, T_q, T_kv]  (only if return_weights=True)
        """
        # 1. Project Q, K, V and reshape to multi-head form
        Q = self._split_heads(self.W_q(query))    # [B, h, T_q,  d_k]
        K = self._split_heads(self.W_k(key))      # [B, h, T_kv, d_k]
        V = self._split_heads(self.W_v(value))    # [B, h, T_kv, d_k]

        # 2. Scaled dot-product scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # [B, h, T_q, T_kv]

        # 3. Apply key-padding mask (mask out <PAD> tokens in source)
        if key_padding_mask is not None:
            # Expand to [B, 1, 1, T_kv] for broadcasting over heads and queries
            mask = key_padding_mask[:, None, None, :]
            scores = scores.masked_fill(mask, float("-inf"))

        # 4. Softmax + dropout
        attn_weights = F.softmax(scores, dim=-1)    # [B, h, T_q, T_kv]
        attn_weights = self.attn_dropout(attn_weights)

        # 5. Weighted sum of values
        context = torch.matmul(attn_weights, V)     # [B, h, T_q, d_k]

        # 6. Merge heads and project to d_model
        context = self._merge_heads(context)        # [B, T_q, d_model]
        output  = self.W_o(context)

        if return_weights:
            return output, attn_weights
        return output


# ──────────────────────────────────────────────────────────────
# 3.  Full Decoder Layer (masked self-attn + cross-attn + FFN)
# ──────────────────────────────────────────────────────────────

class DecoderLayer(nn.Module):
    """
    One Transformer decoder layer (Pre-LN variant):

        x = tgt + drop(SelfAttn(norm(tgt), tgt, tgt, causal_mask))
        x = x   + drop(CrossAttn(norm(x), memory, memory, pad_mask))
        x = x   + drop(FFN(norm(x)))
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int = 2048,
        dropout: float = 0.1,
    ):
        super().__init__()

        # Sub-layer 1: Masked self-attention
        self.self_attn = nn.MultiheadAttention(
            d_model, num_heads, dropout=dropout, batch_first=True
        )

        # Sub-layer 2: Cross-attention
        self.cross_attn = MultiHeadCrossAttention(d_model, num_heads, dropout)

        # Sub-layer 3: Position-wise FFN
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(
        self,
        tgt:    torch.Tensor,                   # [B, T_q,  d_model]
        memory: torch.Tensor,                   # [B, T_kv, d_model]
        tgt_mask: torch.Tensor | None = None,   # [T_q, T_q] causal mask
        memory_key_padding_mask: torch.BoolTensor | None = None,  # [B, T_kv]
    ):
        # 1. Masked self-attention (Pre-LN)
        x = self.norm1(tgt)
        sa, _ = self.self_attn(x, x, x, attn_mask=tgt_mask, need_weights=False)
        tgt = tgt + self.drop(sa)

        # 2. Cross-attention  ← KEY STEP
        x = self.norm2(tgt)
        ca = self.cross_attn(
            query=x,
            key=memory,
            value=memory,
            key_padding_mask=memory_key_padding_mask,
        )
        tgt = tgt + self.drop(ca)

        # 3. Feed-forward
        x = self.norm3(tgt)
        tgt = tgt + self.drop(self.ffn(x))

        return tgt


# ──────────────────────────────────────────────────────────────
# 4.  FLASH ATTENTION variant (PyTorch 2.0+, memory-efficient)
# ──────────────────────────────────────────────────────────────

class FlashCrossAttention(nn.Module):
    """
    Drop-in replacement for MultiHeadCrossAttention that uses
    PyTorch's fused scaled_dot_product_attention (Flash Attention).

    Requires PyTorch >= 2.0.
    """

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.dropout = dropout

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def _split(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, query, key, value, key_padding_mask=None):
        Q = self._split(self.W_q(query))
        K = self._split(self.W_k(key))
        V = self._split(self.W_v(value))

        attn_mask = None
        if key_padding_mask is not None:
            # Convert bool mask [B, T_kv] → float [B, 1, 1, T_kv]
            attn_mask = key_padding_mask[:, None, None, :].float() * -1e9

        # Fused kernel: no explicit attention matrix materialized
        context = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
        )  # [B, heads, T_q, d_k]

        context = context.transpose(1, 2).contiguous().view(query.size(0), -1, self.d_model)
        return self.W_o(context)


# ──────────────────────────────────────────────────────────────
# 5.  DEMO / QUICK TEST
# ──────────────────────────────────────────────────────────────

if __name__ == "__main__":
    torch.manual_seed(42)

    # ── Hyperparameters ──
    B        = 2      # batch size
    T_q      = 5      # target (decoder) sequence length
    T_kv     = 10     # source (encoder) sequence length
    d_model  = 256
    num_heads = 8

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Running on: {device}\n")

    # ── Dummy tensors ──
    decoder_in  = torch.randn(B, T_q,  d_model, device=device)
    encoder_out = torch.randn(B, T_kv, d_model, device=device)

    # Simulate padded source: last 2 positions are <PAD>
    pad_mask = torch.zeros(B, T_kv, dtype=torch.bool, device=device)
    pad_mask[:, -2:] = True   # True = ignore these positions

    # ── Test MultiHeadCrossAttention ──
    mhca = MultiHeadCrossAttention(d_model, num_heads).to(device)
    out, weights = mhca(decoder_in, encoder_out, encoder_out,
                        key_padding_mask=pad_mask, return_weights=True)

    print("=== MultiHeadCrossAttention ===")
    print(f"  Output shape  : {out.shape}")       # [2, 5, 256]
    print(f"  Weight shape  : {weights.shape}")   # [2, 8, 5, 10]
    print(f"  Attn sum (should ≈ 1.0 on non-pad): "
          f"{weights[0, 0, 0, :8].sum().item():.4f}")

    # ── Test DecoderLayer ──
    layer = DecoderLayer(d_model, num_heads).to(device)
    layer_out = layer(decoder_in, encoder_out, memory_key_padding_mask=pad_mask)
    print("\n=== DecoderLayer ===")
    print(f"  Output shape  : {layer_out.shape}")  # [2, 5, 256]

    # ── Test FlashCrossAttention ──
    flash = FlashCrossAttention(d_model, num_heads).to(device)
    flash_out = flash(decoder_in, encoder_out, encoder_out, key_padding_mask=pad_mask)
    print("\n=== FlashCrossAttention ===")
    print(f"  Output shape  : {flash_out.shape}")  # [2, 5, 256]

    # ── NumPy demo ──
    print("\n=== NumPy single-head demo ===")
    Q_np = np.random.randn(T_q, d_model)
    K_np = np.random.randn(T_kv, d_model)
    V_np = np.random.randn(T_kv, d_model)
    np_out, np_w = scaled_dot_product_attention(Q_np, K_np, V_np)
    print(f"  Output shape  : {np_out.shape}")     # (5, 256)
    print(f"  Weights shape : {np_w.shape}")       # (5, 10)
    print(f"  Weights sum   : {np_w.sum(axis=-1)}")  # all ≈ 1.0

    print("\nAll checks passed!")

Running on: cuda

=== MultiHeadCrossAttention ===
  Output shape  : torch.Size([2, 5, 256])
  Weight shape  : torch.Size([2, 8, 5, 10])
  Attn sum (should ≈ 1.0 on non-pad): 1.1111

=== DecoderLayer ===
  Output shape  : torch.Size([2, 5, 256])

=== FlashCrossAttention ===
  Output shape  : torch.Size([2, 5, 256])

=== NumPy single-head demo ===
  Output shape  : (5, 256)
  Weights shape : (5, 10)
  Weights sum   : [1. 1. 1. 1. 1.]

All checks passed!
